# NOVA — terrain-relative reward PROBE (Colab GPU)

Runs the **probe** for the terrain-relative reward fix (PR #130): resume the stairs teacher
under the corrected reward and ask *does it now step UP instead of scraping the riser, without
losing the flat gait?*

**Flow:** config → get code (asserts the fix is present) → deps → drive → GPU sanity →
probe dry-run → probe → judge (rollouts + `climbed` numbers) → export.

Checkpoints live on Drive, so a disconnect costs nothing — re-run the probe cell and it resumes
from its own latest checkpoint (it does **not** re-graft from the teacher).

> Runtime → Change runtime type → **GPU** (T4 is enough) before you start.

## 1. Config — the single source of truth (edit here, nowhere else)

In [ ]:
# Every path/knob the notebook uses. Change them HERE; every cell reads from these.
BRANCH   = "main"          # after PR #130 merges. To test BEFORE merge: "sim/terrain-relative-reward"
DRIVE    = "/content/drive/MyDrive"

# the obs-226 stairs teacher this probe resumes (must already exist on Drive)
TEACHER  = f"{DRIVE}/nova_policy_stairs_final.pkl"

# FRESH probe dir — never reuse nova_stairs_curr (that holds the OLD-reward policy).
CKPT     = f"{DRIVE}/nova_stairs_fix"
POLICY   = f"{DRIVE}/nova_policy_stairs_fix.pkl"   # flat pkl the probe writes every eval

TIMESTEPS = 25_000_000     # ~1.3 h on a T4; kill early via the 5M kill-switch if it's doomed
print("branch", BRANCH, "| ckpt", CKPT)

## 2. GPU + Python check

In [ ]:
import sys
print("Python", sys.version.split()[0])   # want 3.11/3.12 (jax 0.6.0 window)
!nvidia-smi -L || echo "NO GPU -> Runtime > Change runtime type > GPU"

## 3. Get the code — and REFUSE to run stale code

The failure that cost hours twice: `git pull` says *Already up to date* and the run trains against
old code. This cell prints the SHA and **asserts the terrain-relative fix is actually present**;
if it isn't, it raises instead of letting the probe waste a GPU-hour.

In [ ]:
import subprocess, os
%cd /content
if not os.path.isdir("LE_NOVA"):
    !git clone --depth 1 -b {BRANCH} https://github.com/Ace2932/LE_NOVA.git
%cd /content/LE_NOVA
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
%cd /content/LE_NOVA/sim/nova_mjx

sha = subprocess.check_output(["git","rev-parse","--short","HEAD"], text=True).strip()
has_fix = "_terrain_ground_z" in open("env.py").read()
print("HEAD", sha, "| terrain-relative fix present:", has_fix)
assert has_fix, ("env.py has no _terrain_ground_z -> this is PRE-fix code. "
                 "Merge PR #130 (or set BRANCH='sim/terrain-relative-reward') and re-run cell 1+3.")

## 4. Install pinned deps
brax 0.14.2 + jax 0.6.0 is the validated window. If the sanity cell says `cpu`, Runtime → Restart session, re-run from here.

In [ ]:
!pip install -q "jax[cuda12]==0.6.0" brax==0.14.2 "orbax-checkpoint>=0.11.22" \
    "mujoco>=3.10" "mujoco-mjx>=3.10" "imageio>=2.31" imageio-ffmpeg 2>&1 | tail -3
print("deps installed — if the next cell says cpu, restart runtime + re-run from here")

## 5. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.exists(TEACHER), f"teacher pkl missing: {TEACHER}"
print("teacher:", TEACHER)

## 6. Sanity — GPU backend + the fix's own test suite
Runs the terrain-relative test suite (T1-T9 + climb + flat_frac) on GPU. The mj_ray oracle (T8) is the load-bearing check that the reward reads the real collision surface.

In [ ]:
import jax
print("jax backend:", jax.default_backend())
assert jax.default_backend() == "gpu", "not on GPU — restart runtime + re-run cell 4"
!python test_terrain_relative.py 2>&1 | tail -3

## 7. Probe — DRY RUN first

Prints the **reward fingerprint** — terrain 1.00, flat-frac 0.25, the graft source, and (critically)
the git SHA + `(+uncommitted changes)` flag so you can confirm you're on the fixed code — then exits
without training. A single-stage probe shows the fingerprint, not a curriculum plan table.

In [ ]:
# single-stage probe (NOT --curriculum): resume the teacher under the fixed reward.
import glob, os
have_ckpt = any(os.path.basename(d).isdigit()
                for d in glob.glob(f"{CKPT}/**/*", recursive=True))
print("existing probe checkpoint:", have_ckpt, "(True -> will RESUME, not re-graft)")
!python train.py --heightmap --terrain 1.0 --stair-frac 0.6 --flat-frac 0.25 \
    --restore-params-pkl {TEACHER} --ckpt {CKPT} --out {POLICY} \
    --timesteps {TIMESTEPS} --dry-run

## 8. Probe — the real run (~1.3 h)

**First run** grafts from the teacher (`--restore-params-pkl`). **After a dropout**, re-run this
cell: it detects the probe's own checkpoint and RESUMES it (no re-graft, no lost progress).

Watch the per-eval diagnostics line: `climb`/`climb_max` (net / peak base-z gain — should lift off
~0 and rise), `swing` (mean swing-foot height above local ground — should climb toward the 0.08 m
riser), `w_slip`/`airT` (the contact fix engaging). **5M-step kill-switch:** if `swing` hasn't
moved and `w_slip` hasn't engaged by ~5M with `v_loss` plateaued, it's doomed — stop and restart clean.

In [ ]:
import glob, os
have_ckpt = any(os.path.basename(d).isdigit()
                for d in glob.glob(f"{CKPT}/**/*", recursive=True))
if have_ckpt:
    print(">>> RESUMING the probe's own checkpoint (teacher graft skipped)")
    !python train.py --heightmap --terrain 1.0 --stair-frac 0.6 --flat-frac 0.25 \
        --ckpt {CKPT} --out {POLICY} --timesteps {TIMESTEPS}
else:
    print(">>> FIRST run: grafting from the teacher")
    !python train.py --heightmap --terrain 1.0 --stair-frac 0.6 --flat-frac 0.25 \
        --restore-params-pkl {TEACHER} --ckpt {CKPT} --out {POLICY} --timesteps {TIMESTEPS}

## 9. Rebuild the pkl from the newest checkpoint (robust)

`--out` is written atomically every eval, so `{POLICY}` is already valid. This cell is the
authoritative fallback if you reconnected fresh: it rebuilds the pkl from the highest-STEP
checkpoint (by step number, **not** mtime — mtime lies on Drive), shelling out so no jax import
enters the kernel (avoids the os.fork deadlock).

In [ ]:
import glob, os, subprocess
steps = [d for d in glob.glob(f"{CKPT}/**/*", recursive=True)
         if os.path.isdir(d) and os.path.basename(d).isdigit()]
assert steps, f"no checkpoint under {CKPT}"
newest = max(steps, key=lambda d: int(os.path.basename(d)))
print("newest checkpoint:", newest)
# --add-dims 0 : obs 226 -> 226, just extracts params to a flat pkl
subprocess.run(["python","graft_obs.py","--src",newest,"--add-dims","0","--out",POLICY], check=True)
print("wrote", POLICY)

## 10. JUDGE — rollouts + the `climbed` number (the actual verdict)

The acceptance bar (see the design spec). Videos go to **Drive** (survive disconnect).

- **Stairs** `--stair-level 1.0`: acceptance is `climbed >= +0.16 m` and full 601 frames. (The
  terrain's TZ=0.20 ceiling caps relief at two real 8 cm risers — 0.16 m is both of them.)
- **Flat** `--stair-level 0.0 --vx 0.35`: must survive the full 601 frames (today's teacher falls
  at step 470 here).

Read the printed `traveled +X m in x, climbed +X.XX m in z` line from each.

In [ ]:
import subprocess, os
for name, level, vx in [("stairs", 1.0, 0.25), ("flat", 0.0, 0.35)]:
    for ext in ("mp4","gif"):
        out = f"{DRIVE}/probe_{name}.{ext}"
        subprocess.run(["python","rollout.py","--policy",POLICY,"--heightmap",
                        "--stair-level",str(level),"--vx",str(vx),
                        "--steps","600","--out",out], check=False)
        if os.path.exists(out):
            print("video:", out); break
        print("failed:", out, "- trying next format")

## 11. Export for deploy (only once the probe passes)

⚠ This policy is a **privileged teacher** — obs 226 includes the *perfect* heightmap the real
D456/L2 cannot supply. `export_policy.py` emits a 226-input `.npz` that `policy_runner` cannot
fill on the Jetson. Getting to hardware needs real elevation mapping or student distillation onto
proprioception-only obs. The deployable policy is still the flat 105-d one until then.

In [ ]:
!python export_policy.py --policy {POLICY}
!cp -v nova_policy.npz nova_policy.onnx {DRIVE}/ 2>/dev/null; echo "exported to Drive"